In [ ]:
# Import necessary libraries
import os
import re
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from sqlalchemy import create_engine, text
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
from pathlib import Path
from urllib.parse import urljoin
from selenium.webdriver.chrome.options import Options
import pandas as pd
load_dotenv()

In [ ]:
url = "https://books.toscrape.com/catalogue/page-1.html"

r = requests.get(url)
print(r.status_code)

### WebDriver Initialization

In [ ]:
def make_driver():
    chrome_options = Options()

    # "--headless=new" works better with newer Chrome versions
    #chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # Selenium Manager handles ChromeDriver automatically
    return webdriver.Chrome(options=chrome_options)

### Books Extraction

In [ ]:
# Extract only the first catalogue page and determine the total number of pages.

book_names = []
prices = []
in_stocks = []
ratings = []
book_urls = []
book_images = []
categories = []
source_websites = []
scraped_at = []

SOURCE_WEBSITE = "https://books.toscrape.com/"
BOOK_BASE_URL = urljoin(SOURCE_WEBSITE, "catalogue/")
first_page_url = urljoin(BOOK_BASE_URL, "page-1.html")

detail_driver = make_driver()

try:
    detail_driver.get(first_page_url)

    WebDriverWait(detail_driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "article.product_pod"))
    )

    soup = BeautifulSoup(detail_driver.page_source, "lxml")

    # Extract the total number of catalogue pages from "Page 1 of 50".
    pager_element = soup.select_one("li.current")
    pager_text = pager_element.get_text(" ", strip=True) if pager_element else ""
    total_pages = int(pager_text.split()[-1])
    print(f"Total pages: {total_pages}")

    # Extract books from page 1 only.
    books = soup.select("article.product_pod")

    for book in books:
        title_element = book.select_one("h3 a")
        price_element = book.select_one("p.price_color")
        rating_element = book.select_one("p.star-rating")
        image_element = book.select_one("img")

        book_name = title_element.get("title") if title_element else None

        price = (
            price_element.get_text(strip=True).replace("£", "").replace(",", "")
            if price_element
            else None
        )

        rating_classes = rating_element.get("class", []) if rating_element else []
        rating = next(
            (value for value in rating_classes if value != "star-rating"),
            None,
        )

        book_url = (
            urljoin(first_page_url, title_element.get("href"))
            if title_element and title_element.get("href")
            else None
        )

        book_image = (
            urljoin(first_page_url, image_element.get("src"))
            if image_element and image_element.get("src")
            else None
        )

        category = None
        in_stock = None

        # Extract category and availability from the detail page.
        if book_url:
            detail_driver.get(book_url)

            WebDriverWait(detail_driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "ul.breadcrumb"))
            )

            detail_soup = BeautifulSoup(
                detail_driver.page_source,
                "lxml",
            )

            category_element = detail_soup.select_one(
                "ul.breadcrumb li:nth-of-type(3) a"
            )
            category = (
                category_element.get_text(strip=True) if category_element else None
            )

            for row in detail_soup.select("table.table-striped tr"):
                heading = row.select_one("th")
                value = row.select_one("td")

                if heading and heading.get_text(strip=True) == "Availability":
                    availability_text = value.get_text(" ", strip=True) if value else ""
                    stock_match = re.search(r"\d+", availability_text)
                    in_stock = int(stock_match.group()) if stock_match else None
                    break

        book_names.append(book_name)
        prices.append(price)
        in_stocks.append(in_stock)
        ratings.append(rating)
        categories.append(category)
        book_urls.append(book_url)
        book_images.append(book_image)
        source_websites.append(SOURCE_WEBSITE)
        scraped_at.append(pd.Timestamp.now(tz="UTC"))

    print(f"Scraped page 1: {len(book_names)} books collected")

finally:
    detail_driver.quit()


data = {
    "book_names": book_names,
    "availability": in_stocks,
    "ratings": ratings,
    "prices": prices,
    "categories": categories,
    "book_urls": book_urls,
    "book_images": book_images,
    "source_website": source_websites,
    "scraped_at": scraped_at,
}

books_df = pd.DataFrame(data)

print(f"Catalogue pages available: {total_pages}")
print(f"Books extracted from page 1: {len(books_df)}")
books_df

In [ ]:
# Lists to store the extracted data
book_names = []
prices = []
in_stocks = []
ratings = []
book_urls = []
book_images = []
categories = []
source_websites = []
scraped_at = []


SOURCE_WEBSITE = "https://books.toscrape.com/"
BOOK_BASE_URL = urljoin(SOURCE_WEBSITE, "catalogue/")
current_url = urljoin(BOOK_BASE_URL, "page-1.html")
page_number = 1


detail_driver = make_driver()
try:
    while current_url:
        url = current_url
        detail_driver.get(url)
        WebDriverWait(detail_driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article.product_pod"))
        )

        # Parse the catalogue page currently displayed in Chrome.
        soup = BeautifulSoup(detail_driver.page_source, "lxml")

        # Locate each book card in the catalogue.
        books = soup.select("article.product_pod")

        # Save the next page URL before visiting book detail pages.
        next_element = soup.select_one("li.next a")
        next_page_url = (
            urljoin(url, next_element.get("href"))
            if next_element and next_element.get("href")
            else None
        )

        for book in books:
            # Extract book name
            title_element = book.select_one("h3 a")
            price_element = book.select_one("p.price_color")
            rating_element = book.select_one("p.star-rating")

            book_name = (
                title_element.get("title") if title_element else None
            )

            # Extract prices
            price = (
                price_element.get_text(strip=True)
                .replace("£", "")
                .replace(",", "")
                if price_element
                else None
            )

            # The stock quantity is extracted from the detail page below.
            in_stock = None

            # Extract rating
            rating_classes = (
                rating_element.get("class", []) if rating_element else []
            )
            rating = next(
                (value for value in rating_classes if value != "star-rating"),
                None,
            )

            # Extract book URL
            book_url = (
                urljoin(BOOK_BASE_URL, title_element["href"])
                if title_element and title_element.get("href")
                else None
            )

            # Category and full availability are on the book detail page.
            category = None
            if book_url:
                detail_driver.get(book_url)
                WebDriverWait(detail_driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "ul.breadcrumb"))
                )
                detail_soup = BeautifulSoup(detail_driver.page_source, "lxml")
                category_element = detail_soup.select_one(
                    "ul.breadcrumb li:nth-of-type(3) a"
                )
                category = (
                    category_element.get_text(strip=True)
                    if category_element
                    else None
                )

                # Extract the quantity, for example: In stock (22 available).
                for row in detail_soup.select("table.table-striped tr"):
                    heading = row.select_one("th")
                    value = row.select_one("td")
                    if heading and heading.get_text(strip=True) == "Availability":
                        availability_text = (
                            value.get_text(" ", strip=True) if value else ""
                        )
                        match = re.search(r"\d+", availability_text)
                        in_stock = int(match.group()) if match else None
                        break

            # Extract book image URL
            book_image = (
                urljoin(BOOK_BASE_URL, book.select_one("img")["src"])
                if book.select_one("img")
                else None
            )

            # Append the extracted data to the respective lists
            book_names.append(book_name)
            prices.append(price)
            in_stocks.append(in_stock)
            ratings.append(rating)
            categories.append(category)
            book_urls.append(book_url)
            book_images.append(book_image)
            source_websites.append(SOURCE_WEBSITE)
            scraped_at.append(pd.Timestamp.now(tz="UTC"))

        print(f"Scraped page {page_number}: {len(book_names)} books collected")
        current_url = next_page_url
        page_number += 1
finally:
    detail_driver.quit()

# Create a DataFrame from the extracted data
data = {
    "book_names": book_names,
    "availability": in_stocks,
    "ratings": ratings,
    "prices": prices,
    "categories": categories,
    "book_urls": book_urls,
    "book_images": book_images,
    "source_website": source_websites,
    "scraped_at": scraped_at,
}

books_df = pd.DataFrame(data)

### Transformation

In [ ]:
# Keep a convenient latest file and an immutable snapshot for each run.
raw_file = Path("../data/raw_data/books_data.csv")
raw_archive_dir = raw_file.parent / "archive"
raw_archive_dir.mkdir(parents=True, exist_ok=True)
raw_timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
raw_snapshot = raw_archive_dir / f"books_data_{raw_timestamp}.csv"
books_df.to_csv(raw_snapshot, index=False)
books_df.to_csv(raw_file, index=False)
print(f"Raw snapshot preserved at {raw_snapshot}")

In [ ]:
# Convert price and availability values to numbers.
books_df["prices"] = pd.to_numeric(books_df["prices"], errors="coerce")
books_df["availability"] = pd.to_numeric(
    books_df["availability"], errors="coerce"
).astype("Int64")

In [ ]:
# Convert the rating column to numeric values for easier analysis
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}
books_df["ratings"] = books_df["ratings"].map(rating_mapping).astype("Int64")

In [ ]:
# Store all categories discovered in the book breadcrumbs.
books_df["categories"] = books_df["categories"].astype("category")

# Keep the source as text and normalize scrape timestamps to UTC.
books_df["source_website"] = (
    books_df["source_website"].astype("string").str.strip()
)
books_df["scraped_at"] = pd.to_datetime(
    books_df["scraped_at"], errors="coerce", format="mixed", utc=True
)

# Check whether the DataFrame is empty.
if len(books_df) == 0:
    raise ValueError("The dataset is empty")

# Count all missing values.
if books_df.isnull().sum().sum() > 0:
    raise ValueError("The dataset contains missing values")

# Count duplicate book URLs.
if books_df["book_urls"].duplicated().sum() > 0:
    raise ValueError("Duplicate book URLs found")

# Find ratings outside the range 1 to 5.
invalid_ratings = books_df[
    (books_df["ratings"] < 1) | (books_df["ratings"] > 5)
]

if len(invalid_ratings) > 0:
    raise ValueError("Ratings must be between 1 and 5")

# Find negative prices.
negative_prices = books_df[books_df["prices"] < 0]

if len(negative_prices) > 0:
    raise ValueError("Prices cannot be negative")

# Find negative availability values.
negative_availability = books_df[books_df["availability"] < 0]

if len(negative_availability) > 0:
    raise ValueError("Availability cannot be negative")

print("Data validation passed")

In [ ]:
# Save the cleaned DataFrame to a CSV file
books_df.to_csv("../data/cleaned_data/books_data.csv", index=False)

### Create Database and Load Books

In [ ]:
# Create a database in PostgreSQL and store the data in a table
# Define the database connection parameters
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

In [ ]:
# Create the PostgreSQL database if it does not already exist
admin_engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/postgres",
    isolation_level="AUTOCOMMIT",
)
with admin_engine.connect() as connection:
    database_exists = connection.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :db_name"),
        {"db_name": db_name},
    ).scalar()
    if not database_exists:
        quoted_db_name = admin_engine.dialect.identifier_preparer.quote_identifier(
            db_name
        )
        connection.exec_driver_sql(f"CREATE DATABASE {quoted_db_name}")

admin_engine.dispose()
engine = create_engine(
    f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

In [ ]:
# # Create schema and table in the database
with engine.connect() as connection:
    # Create schema if it doesn't exist
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS pagepulse;"))

    # Create table if it doesn't exist
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS pagepulse.books (
            book_names TEXT ,
            availability INTEGER,
            ratings TEXT,
            prices FLOAT,
            categories TEXT,
            book_urls TEXT,
            book_images TEXT,
            source_website TEXT,
            scraped_at TIMESTAMPTZ
        );
    """))

In [ ]:
# Load the saved cleaned data into the PostgreSQL database
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS pagepulse"))
    books_df.to_sql(
        "books",
        connection,
        schema="pagepulse",
        if_exists="replace",
        index=False,
    )
print("Data loaded into PostgreSQL database successfully.")

### SQL Business Analysis

In [ ]:
sql_task_1 = """
-- 1. Total number of books
SELECT COUNT(*) AS total_books
FROM pagepulse.books;
"""
result = pd.read_sql(sql_task_1, engine)
result

In [ ]:
sql_task_2 = """
-- 2. Average book price rounded to two decimal places
SELECT ROUND(CAST(AVG(prices) AS numeric), 2) AS average_price
FROM pagepulse.books;
"""
result = pd.read_sql(sql_task_2, engine)
result

In [ ]:
sql_task_3 = """
SELECT book_names, prices FROM pagepulse.books
ORDER BY prices DESC
LIMIT 10;
"""
result = pd.read_sql(sql_task_3, engine)
result

In [ ]:
sql_task_4 = """
-- 4. Largest category by number of books
SELECT categories, COUNT(*) AS book_count
FROM pagepulse.books
GROUP BY categories
ORDER BY book_count DESC
LIMIT 1;
"""
result = pd.read_sql(sql_task_4, engine)
result

In [ ]:
sql_task_5 = """
-- 5. Category with the highest average price
SELECT categories, AVG(prices) AS average_price  
FROM pagepulse.books
GROUP BY categories
ORDER BY average_price DESC
LIMIT 1;
"""
result = pd.read_sql(sql_task_5, engine)
result

In [ ]:
sql_task_6 = """
-- 6. Number of books by rating
SELECT ratings, COUNT(*) AS book_count
FROM pagepulse.books
GROUP BY ratings
ORDER BY book_count DESC;
"""
result = pd.read_sql(sql_task_6, engine)
result

In [ ]:
sql_task_7 = """
-- 7. All books currently in stock
SELECT COUNT(*) AS in_stock_books
FROM pagepulse.books;
"""
result = pd.read_sql(sql_task_7, engine)
result

In [ ]:
sql_task_8 = """
-- 8. Books priced above £ 40
SELECT book_names, prices
FROM pagepulse.books
WHERE prices > 40
ORDER BY prices DESC; 
"""
result = pd.read_sql(sql_task_8, engine)
result

In [ ]:
sql_task_9 = """
-- 9. Average rating by category
SELECT categories, ROUND(AVG(ratings), 2) AS average_rating
FROM pagepulse.books
GROUP BY categories
ORDER BY average_rating DESC; 
"""
result = pd.read_sql(sql_task_9, engine)
result

In [ ]:
sql_task_10 = """
-- 10. Categories with more than 20 books
SELECT categories, COUNT(*) AS book_count
FROM pagepulse.books
GROUP BY categories
HAVING COUNT(*) > 20
ORDER BY book_count DESC;
"""
result = pd.read_sql(sql_task_10, engine)
result

### Business Insights

##### 1. The catalogue has substantial category coverage

**Evidence:** The catalogue contains **1,000 books across 50 categories**.

**What it means:** BookSphere Analytics has enough books to compare categories by size, price, rating, and stock, but results for smaller categories may be less reliable.

#### 2. The typical book is priced at about £35

**Evidence:** The average book price is **£35.07**. A total of **403 books (40.3%)** cost more than £40.

**What it means:** The catalogue has a strong mid-to-premium price position. BookSphere Analytics could use £35.07 as an initial pricing benchmark and investigate whether the catalogue needs more affordable products for price-sensitive customers.

#### 3. The highest-priced products are tightly grouped

**Evidence:** *The Perfect Play (Play by Play #1)* is the most expensive book at **£59.99**. The next four most expensive books cost between **£59.90 and £59.98**.

**What it means:** The premium end of the catalogue appears to have a price ceiling near £60. BookSphere Analytics could monitor whether these products also achieve strong ratings or sales before recommending premium promotions.


#### 4. Nonfiction is the largest clearly defined category

**Evidence:** Category Default (152 books), **Nonfiction contains 110 books**, followed by Sequential Art (75). The Add a comment category contains 67 books

**What it means:** Nonfiction is commercially important because it represents a large portion of the usable assortment. The 219 books (Default plus Add a comment) assigned to ambiguous labels should be reviewed because weak classification can reduce the accuracy of category reporting and recommendations.


#### 5. Lower-rated books slightly outweigh highly rated books

**Evidence:** There are **422 one- or two-star books (42.2%)**, compared with **375 four- or five-star books (37.5%)**. One-star books form the largest individual rating group, with 226 titles.

**What it means:** BookSphere Analytics may need to identify categories containing a high share of poorly rated products. These titles could be reviewed for replacement or reduced promotion, although rating results should be combined with sales and customer-engagement data before making assortment decisions.

### Student Reflection

#### 1. What was the most difficult stage?
Avoid hardcoded of number of pages to be scrapped. This is need to ensure the etl pipeline did not break once new number of books are added which in turn increase the number of pages beyond the hardcoded value - say in the case of 50 pages. 

#### 2. What data-quality issue did you encounter?
One data-quality issue was that prices and ratings were scraped as text rather than analysis-ready numeric values. Prices contained the pound symbol (for example, £51.77), while ratings were words such as Three or One. I cleaned the data by removing the currency symbol, converting prices to decimals, and mapping rating words to integers from 1 to 5. I also validated required fields, rating ranges, non-negative values, and duplicate book URLs before loading the data into PostgreSQL.

#### 3. What is the difference between static and dynamic scraping?
Static scraping retrieves data directly from the HTML returned by the server, using tools such as requests and Beautiful Soup. Dynamic scraping uses a browser automation tool such as Selenium when JavaScript or user interaction is required to load the content. In this project, Selenium controls the browser, while Beautiful Soup parses the resulting HTML and extracts the book data.

#### 4. Why should raw data be preserved?
Raw data should be preserved because it provides an unchanged record of what was originally extracted. It allows the cleaning process to be audited, errors to be investigated, and the data to be transformed again without repeating the web scrape. In this project, the raw CSV is kept separately from the cleaned dataset for traceability and reproducibility.

#### 5. What would you improve for production?
For production, I would schedule the pipeline with a workflow orchestrator such as Airflow, add automated tests, structured logging, retries, monitoring, and failure alerts. I would also run the scraper in a container, store credentials securely, respect rate limits, and load data incrementally using database upserts instead of replacing the whole table. Finally, I would track data-quality metrics and retain timestamped raw data so every run is auditable and reproducible.

### GitHub Repository URL
https://github.com/esodevops/pagepulse-booksite-etl-pipeline